In [ ]:
%run ./_common

In [ ]:
import warnings, logging, random
warnings.filterwarnings("ignore")

import numpy as np
np.set_printoptions(precision=2)
logging.getLogger("tensorflow").setLevel(logging.ERROR)

In [ ]:
SA = SWSAcademyHelper()
SA.init()

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.functions import vector_to_array
from databricks.feature_engineering import FeatureEngineeringClient
import mlflow

print("INITIALIZING MODULE 3: Data Cleaning & Feature Engineering (Price Movement)")
print("*" * 90)

In [ ]:
# Load the source table ONCE (no drift simulation here)

yahoo_finance_table = SA.datasets.yahoo.table
raw_df = spark.table(yahoo_finance_table)

print(f"Loaded source table: {yahoo_finance_table}")
print(f" - Rows: {raw_df.count()}")
print(f" - Cols: {len(raw_df.columns)}")

In [ ]:
#  Minimal cleanup & helpers (safe math only)
#    - Normalize 'null'/'N/A' → real nulls
#    - Cast numerics
#    - Parse BID/ASK → price/size
#    - Split DAY/WEEK ranges → low/high

CANON_NULLS = ["null", "NULL", "N/A", "n/a", "", "NaN"]

def canon_null(col):
    return F.when(F.col(col).isin(CANON_NULLS), F.lit(None)).otherwise(F.col(col))

def to_double(col):
    return F.regexp_replace(canon_null(col), ",", "").cast(DoubleType())

def to_long(col):
    return F.regexp_replace(canon_null(col), ",", "").cast(LongType())

def parse_px_sz(col):
    # "1.7550 x 100000" → (price, size)
    price_s = F.trim(F.regexp_extract(canon_null(col), r"^\s*([0-9.,]+)", 1))
    size_s  = F.trim(F.regexp_extract(canon_null(col), r"x\s*([0-9,]+)\s*$", 1))
    price_d = F.regexp_replace(price_s, ",", "").cast(DoubleType())
    size_l  = F.regexp_replace(size_s,  ",", "").cast(LongType())
    return price_d.alias(f"{col.lower()}_price"), size_l.alias(f"{col.lower()}_size")

def parse_range(col):
    # "1,195.05 - 2,195.05" → (low, high)
    low_s  = F.trim(F.regexp_extract(canon_null(col), r"^\s*([0-9.,-]+)", 1))
    high_s = F.trim(F.regexp_extract(canon_null(col), r"-\s*([0-9.,-]+)\s*$", 1))
    low_d  = F.regexp_replace(low_s,  ",", "").cast(DoubleType())
    high_d = F.regexp_replace(high_s, ",", "").cast(DoubleType())
    return low_d.alias(f"{col.lower()}_low"), high_d.alias(f"{col.lower()}_high")

# Cast & normalize key numeric columns
base = (
    raw_df
    .withColumn("CLOSING_PRICE_d",   to_double("CLOSING_PRICE"))
    .withColumn("PREVIOUS_CLOSE_d",  to_double("PREVIOUS_CLOSE"))
    .withColumn("OPEN_d",            to_double("OPEN"))
    .withColumn("VOLUME_d",          to_long("VOLUME"))
)

# Parse BID/ASK → price & size
bid_price, bid_size = parse_px_sz("BID")
ask_price, ask_size = parse_px_sz("ASK")

# Parse DAY/WEEK ranges → low/high
day_low, day_high   = parse_range("DAY_RANGE")
week_low, week_high = parse_range("WEEK_RANGE")

feat = (
    base
    .select(
        "NAME", "COMPANY_ID", "STOCK_TICKER",           # identifiers (kept for joins)
        "ENTITY_TYPE", "CURRENCY", "EXCHANGE",          # categoricals (kept for joins/encoding if needed)
        "CLOSING_PRICE_d", "PREVIOUS_CLOSE_d", "OPEN_d", "VOLUME_d",
        "BID", "ASK", "DAY_RANGE", "WEEK_RANGE"
    )
    .withColumns({
        "bid_price": bid_price, "bid_size": bid_size,
        "ask_price": ask_price, "ask_size": ask_size,
        "day_low": day_low,     "day_high": day_high,
        "week_low": week_low,   "week_high": week_high,
    })
    # has_* flags
    .withColumns({
        "has_bidask":     F.when(F.col("bid_price").isNotNull()  & F.col("ask_price").isNotNull(),  F.lit(1)).otherwise(F.lit(0)),
        "has_day_range":  F.when(F.col("day_low").isNotNull()    & F.col("day_high").isNotNull(),   F.lit(1)).otherwise(F.lit(0)),
        "has_week_range": F.when(F.col("week_low").isNotNull()   & F.col("week_high").isNotNull(),  F.lit(1)).otherwise(F.lit(0)),
    })
    # Compact engineered features (guard denominators; no NaN/Inf)
    .withColumns({
        "open_gap": F.when((F.col("PREVIOUS_CLOSE_d").isNull()) | (F.col("PREVIOUS_CLOSE_d") == 0), F.lit(0.0))
                     .otherwise( (F.col("OPEN_d") - F.col("PREVIOUS_CLOSE_d")) / F.col("PREVIOUS_CLOSE_d") ),

        "bid_ask_spread": F.when(F.col("ask_price").isNull() | F.col("bid_price").isNull(), F.lit(0.0))
                           .otherwise( F.col("ask_price") - F.col("bid_price") ),

        "relative_spread": F.when(
                               (F.col("PREVIOUS_CLOSE_d").isNull()) | (F.col("PREVIOUS_CLOSE_d") == 0) |
                               F.col("ask_price").isNull() | F.col("bid_price").isNull(),
                               F.lit(0.0)
                           ).otherwise( (F.col("ask_price") - F.col("bid_price")) / F.col("PREVIOUS_CLOSE_d") ),

        "day_range_pct": F.when(
                              (F.col("PREVIOUS_CLOSE_d").isNull()) | (F.col("PREVIOUS_CLOSE_d") == 0) |
                              F.col("day_high").isNull() | F.col("day_low").isNull(),
                              F.lit(0.0)
                          ).otherwise( (F.col("day_high") - F.col("day_low")) / F.col("PREVIOUS_CLOSE_d") ),

        "week_range_pct": F.when(
                               (F.col("PREVIOUS_CLOSE_d").isNull()) | (F.col("PREVIOUS_CLOSE_d") == 0) |
                               F.col("week_high").isNull() | F.col("week_low").isNull(),
                               F.lit(0.0)
                           ).otherwise( (F.col("week_high") - F.col("week_low")) / F.col("PREVIOUS_CLOSE_d") ),

        "book_imbalance": F.when(F.col("bid_size").isNull() | F.col("ask_size").isNull(), F.lit(0.0))
                           .otherwise( (F.col("bid_size") - F.col("ask_size")) / (F.col("bid_size") + F.col("ask_size") + F.lit(1e-9)) ),
    })
    # Label: UP (1) if close > previous close, else 0
    .withColumn("label_up", F.when(F.col("CLOSING_PRICE_d") > F.col("PREVIOUS_CLOSE_d"), F.lit(1)).otherwise(F.lit(0)))
)

In [ ]:
# Final prepared frame for modeling (no leakage)
#    - Exclude CLOSING_PRICE from features
#    - Keep identifiers for joins/inspection

ID_COLS = ["COMPANY_ID", "STOCK_TICKER", "NAME"]
CAT_COLS = ["ENTITY_TYPE", "CURRENCY", "EXCHANGE"]  # retained for joins/optional encoding
FEATURE_COLS = [
    "OPEN_d", "VOLUME_d",
    "bid_price", "ask_price", "bid_size", "ask_size",
    "day_low", "day_high", "week_low", "week_high",
    "open_gap", "bid_ask_spread", "relative_spread",
    "day_range_pct", "week_range_pct", "book_imbalance",
    "has_bidask", "has_day_range", "has_week_range"
]

prepared_df = feat
# Coalesce numeric features to safe defaults (0.0)
for c in FEATURE_COLS:
    prepared_df = prepared_df.withColumn(c, F.coalesce(F.col(c), F.lit(0.0)))

prepared_df = prepared_df.select(*(ID_COLS + CAT_COLS + FEATURE_COLS + ["label_up"]))

#print("Prepared features preview (subset of columns):")
#display(prepared_df.limit(10).select(*(ID_COLS + ["label_up"]) + FEATURE_COLS[:8]))
print("\nColumn sets for downstream steps:")
print(" - ID_COLS      :", ID_COLS)
print(" - CAT_COLS     :", CAT_COLS, "(kept for later joins/encoding)")
print(" - FEATURE_COLS :", FEATURE_COLS)
print(" - LABEL        : label_up (1 if close > previous close)")

print("\nSanity checks:")
rows = prepared_df.count()
pos = prepared_df.agg(F.sum("label_up")).first()[0]
print(f" - Rows: {rows} | Positives (label_up=1): {int(pos) if pos is not None else 0}")

In [ ]:
# Assemble features_array (array<double>), NaN-safe

assembler = VectorAssembler(inputCols=FEATURE_COLS, outputCol="features", handleInvalid="keep")
assembled = assembler.transform(prepared_df)

final_features_df = (
    assembled
    .withColumn("features_array_raw", vector_to_array("features"))
    .withColumn(
        "features_array",
        F.expr("transform(features_array_raw, x -> coalesce(nanvl(x, double(0.0)), double(0.0)))")
    )
    .withColumn("FEATURE_ID", F.monotonically_increasing_id())
    .select(
        "FEATURE_ID",
        "COMPANY_ID", "STOCK_TICKER",  # useful identifiers to keep
        "label_up",
        "features_array"
    )
)

#print("\nPreview of training-ready features (features_array + label_up):")
#display(final_features_df.limit(10))

# Prove array has no NaN/null
_bad = final_features_df.selectExpr(
    "exists(features_array, x -> x is null OR isnan(x)) AS has_bad"
).agg(F.max("has_bad")).first()[0]
print(f"Array NaN/null present? {_bad}")

In [ ]:
# Register Feature Store table (Unity Catalog)

fe = FeatureEngineeringClient()
features_table_name = f"{SA.catalog_name}.{SA.schema_name}.price_move_features"

print(f"\nCreating (or replacing) Feature Store table: {features_table_name}")
try:
    fe.drop_table(name=features_table_name)
except Exception as e:
    print(f"(Skip drop if not found) {e}")

fe.create_table(
    name=features_table_name,
    primary_keys=["FEATURE_ID"],
    df=final_features_df,
    description="Price-movement lab features: features_array (array<double>) + label_up, engineered from Yahoo Finance Business table."
)

In [ ]:
# Optional: traceability in MLflow
with mlflow.start_run(run_name="module3_price_move_fs_registration", nested=True):
    mlflow.log_param("fs_table_name", features_table_name)
    mlflow.log_param("primary_key", "FEATURE_ID")
    mlflow.log_param("label_col", "label_up")
    mlflow.log_param("source_table", yahoo_finance_table)

In [ ]:
# Read-back sanity check
df_fs = fe.read_table(name=features_table_name)
#print("\nFeature Store table sample:")
#display(df_fs.limit(10))

In [ ]:
rows = df_fs.count()
avg_len = df_fs.selectExpr("aggregate(features_array, 0, (acc, _) -> acc + 1) as L").agg(F.avg("L")).first()[0]
#print(f"FS QA — rows: {rows}, avg feature length: {avg_len:.1f}")

#print("\nLabel distribution (label_up):")
#display(df_fs.groupBy("label_up").count().orderBy(F.desc("count")))

In [ ]:
print("\n" + "="*90)
print("CLASSROOM SETUP COMPLETE — Cleaned & engineered features (no leakage) are ready for Ray/MLflow.")
print("="*90)

print("\nArtifacts")
print(f" - Dataset Source:          {yahoo_finance_table}")
print(f" - Feature Store Table:     {features_table_name}")
print(f" - Primary Key:             FEATURE_ID")
print(f" - Label (binary):          label_up")
print(f" - Column:                  features_array (array<double>)")

# (Optional) Export pointers for main lab notebook
SA.prepared = {
    "SRC_TABLE": yahoo_finance_table,
    "ID_COLS": ID_COLS,
    "CAT_COLS": CAT_COLS,
    "FEATURE_COLS": FEATURE_COLS,
    "prepared_df": prepared_df,
    "FS_TABLE_NAME": features_table_name
}